In [4]:


import pandas as pd
import random
import json
from datetime import datetime


LIMITES = { 
    "temp_interna_min": -10,
    "temp_interna_max": 40,
    "temp_externa_min": -80,
    "temp_externa_max": 60,
    "integridade_estrutural": 1,
    "nivel_energia_min": 85,
    "pressao_tanque_min": 200,
    "pressao_tanque_max": 350,
    "modulos_criticos_ok": True,
}


def coletar_dados():
    dados = {
        "timestamp": datetime.now().isoformat(),
        "temp_interna": float(input("Digite a temperatura interna (°C): ")),
        "temp_externa": float(input("Digite a temperatura externa (°C): ")),
        "integridade_estrutural": int(input("Digite a integridade estrutural (1 para OK, 0 para falha): ")),
        "nivel_energia": float(input("Digite o nível de energia (%): ")),
        "pressao_tanque_ox": float(input("Digite a pressão do tanque de oxigênio (kPa): ")),
        "pressao_tanque_comb": float(input("Digite a pressão do tanque de combustível (kPa): ")),
        "modulos": {
            "propulsao": bool(int(input("Digite o status do módulo de propulsão (1 para OK, 0 para falha): "))),
            "navegacao": bool(int(input("Digite o status do módulo de navegação (1 para OK, 0 para falha): "))),
            "comunicacao": bool(int(input("Digite o status do módulo de comunicação (1 para OK, 0 para falha): "))),
            "controle_termico": bool(int(input("Digite o status do módulo de controle térmico (1 para OK, 0 para falha): "))),
            "separacao_estagios": bool(int(input("Digite o status da separação de estágios (1 para OK, 0 para falha): ")))
        }
    }
    return dados


def verificar_dados(dados):
    alertas = []
    verificacoes = {}

    ti = dados["temp_interna"]
    ok_ti = LIMITES["temp_interna_min"] <= ti <= LIMITES["temp_interna_max"]
    verificacoes["temp_interna"] = ok_ti
    if not ok_ti:
        alertas.append(f"[ALERTA] Temperatura interna fora do limite: {ti}°C "
                      f"(esperado: {LIMITES['temp_interna_min']} a {LIMITES['temp_interna_max']}°C)")

    te = dados["temp_externa"]
    ok_te = LIMITES["temp_externa_min"] <= te <= LIMITES["temp_externa_max"]
    verificacoes["temp_externa"] = ok_te
    if not ok_te:
        alertas.append(f"[ALERTA] Temperatura externa fora do limite: {te}°C "
                      f"(esperado: {LIMITES['temp_externa_min']} a {LIMITES['temp_externa_max']}°C)")

    ie = dados["integridade_estrutural"]
    ok_ie = ie == LIMITES["integridade_estrutural"]
    verificacoes["integridade_estrutural"] = ok_ie
    if not ok_ie:
        alertas.append("[CRÍTICO] Integridade estrutural comprometida! (valor=0)")

    ne = dados["nivel_energia"]
    ok_ne = ne >= LIMITES["nivel_energia_min"]
    verificacoes["nivel_energia"] = ok_ne
    if not ok_ne:
        alertas.append(f"[ALERTA] Nível de energia insuficiente: {ne}% "
                      f"(mínimo: {LIMITES['nivel_energia_min']}%)")

    po = dados["pressao_tanque_ox"]
    pc = dados["pressao_tanque_comb"]
    ok_po = LIMITES["pressao_tanque_min"] <= po <= LIMITES["pressao_tanque_max"]
    ok_pc = LIMITES["pressao_tanque_min"] <= pc <= LIMITES["pressao_tanque_max"]
    verificacoes["pressao_tanque_ox"] = ok_po
    verificacoes["pressao_tanque_comb"] = ok_pc
    if not ok_po:
        alertas.append(f"[CRÍTICO] Pressão tanque oxidante fora do limite: {po} kPa")
    if not ok_pc:
        alertas.append(f"[CRÍTICO] Pressão tanque combustível fora do limite: {pc} kPa")

    modulos_falhos = [m for m, status in dados["modulos"].items() if not status]
    ok_modulos = len(modulos_falhos) == 0
    verificacoes["modulos_criticos"] = ok_modulos
    if not ok_modulos:
        alertas.append(f"[CRÍTICO] Módulos com falha: {', '.join(modulos_falhos)}")

    todos_ok = all(verificacoes.values())
    decisao = "PRONTO PARA DECOLAR" if todos_ok else "DECOLAGEM ABORTADA"

    return {
        "verificacoes": verificacoes,
        "alertas": alertas,
        "decisao": decisao,
        "aprovado": todos_ok
    }


def analise_energetica(nivel_energia_pct):
    capacidade_total_kwh = 500.0
    consumo_decolagem_kwh = 120.0
    fator_perdas = 0.08

    carga_atual_kwh = capacidade_total_kwh * (nivel_energia_pct / 100)
    perdas_kwh = consumo_decolagem_kwh * fator_perdas
    consumo_real = consumo_decolagem_kwh + perdas_kwh
    autonomia_kwh = carga_atual_kwh - consumo_real
    suficiente = autonomia_kwh > 0

    return {
        "capacidade_total_kwh": capacidade_total_kwh,
        "carga_atual_kwh": round(carga_atual_kwh, 2),
        "consumo_decolagem_kwh": consumo_decolagem_kwh,
        "perdas_kwh": round(perdas_kwh, 2),
        "consumo_real_kwh": round(consumo_real, 2),
        "autonomia_pos_decolagem_kwh": round(autonomia_kwh, 2),
        "suficiente": suficiente
    }


def analise_ia(dados, resultado_verificacao, energia):
    classificacoes = []
    anomalias = []
    sugestoes = []

    ti = dados["temp_interna"]
    if ti < 0:
        classificacoes.append("Temperatura interna: FRIA (risco de condensação em circuitos)")
        anomalias.append("Temperatura interna abaixo de zero — risco de falha em sensores")
        sugestoes.append("Ativar sistema de aquecimento interno antes da decolagem")
    elif ti <= 30:
        classificacoes.append("Temperatura interna: NOMINAL")
    else:
        classificacoes.append("Temperatura interna: ELEVADA (monitorar resfriamento)")
        sugestoes.append("Verificar sistema de refrigeração interno")

    ne = dados["nivel_energia"]
    if ne >= 95:
        classificacoes.append("Energia: EXCELENTE")
    elif ne >= 85:
        classificacoes.append("Energia: ADEQUADA")
    else:
        classificacoes.append("Energia: INSUFICIENTE — risco operacional")
        anomalias.append("Nível de energia abaixo do mínimo recomendado")
        sugestoes.append("Recarregar bancos de energia antes da tentativa de decolagem")

    po = dados["pressao_tanque_ox"]
    pc = dados["pressao_tanque_comb"]
    if po < 210 or pc < 210:
        anomalias.append("Pressão em tanques próxima ao limite inferior — risco de falha de propulsão")
        sugestoes.append("Pressurizar tanques ao nível nominal (270–300 kPa)")
    elif po > 330 or pc > 330:
        anomalias.append("Pressão em tanques próxima ao limite superior — risco de ruptura")
        sugestoes.append("Liberar pressão excedente via válvula de alívio")
    else:
        classificacoes.append("Pressão dos tanques: NOMINAL")

    falhos = [m for m, s in dados["modulos"].items() if not s]
    if falhos:
        anomalias.append(f"Módulo(s) crítico(s) com falha detectada: {', '.join(falhos)}")
        sugestoes.append("Executar diagnóstico completo e substituir módulo(s) defeituoso(s)")
    else:
        classificacoes.append("Todos os módulos críticos: OPERACIONAIS")

    if dados["integridade_estrutural"] == 0:
        anomalias.append("Integridade estrutural comprometida — possível dano físico à estrutura")
        sugestoes.append("Inspeção visual e ultrassônica completa antes de qualquer nova tentativa")

    if not resultado_verificacao["aprovado"]:
        sugestoes.append("Missão deve ser adiada até resolução de todos os alertas críticos")
    else:
        sugestoes.append("Todos os parâmetros nominais — missão pode prosseguir conforme planejado")

    return {
        "classificacoes": classificacoes,
        "anomalias": anomalias if anomalias else ["Nenhuma anomalia detectada"],
        "sugestoes": sugestoes
    }


def gerar_relatorio(dados, resultado, energia, ia):
    sep = "=" * 60
    print(f"\n{sep}")
    print(" RELATORIO DE VERIFICAÇÃO")
    print(f"  {dados['timestamp']}")
    print(sep)

    print("\n DADOS COLETADOS")
    print(f"  Temperatura Interna  : {dados['temp_interna']} °C")
    print(f"  Temperatura Externa  : {dados['temp_externa']} °C")
    print(f"  Integridade Estrut.  : {'OK (1)' if dados['integridade_estrutural'] else 'FALHA (0)'}")
    print(f"  Nível de Energia     : {dados['nivel_energia']} %")
    print(f"  Pressão Tanque Ox.   : {dados['pressao_tanque_ox']} kPa")
    print(f"  Pressão Tanque Comb. : {dados['pressao_tanque_comb']} kPa")
    print("  Módulos Críticos:")
    for modulo, status in dados["modulos"].items():
        print(f"   {modulo.replace('_', ' ').title()} : {'OK (1)' if status else 'FALHA (0)'}")

    print("\nVERIFICAÇÕES DE SEGURANÇA")
    for check, ok in resultado["verificacoes"].items():
        print(f"  {check.replace('_', ' ').title()} : {'OK' if ok else 'FALHA'}")

    if resultado["alertas"]:
        print("\n ALERTAS")
        for a in resultado["alertas"]:
            print(f"  {a}")

    print("\n ANÁLISE ENERGÉTICA")
    e = energia
    print(f"  Capacidade Total     : {e['capacidade_total_kwh']} kWh")
    print(f"  Carga Atual          : {e['carga_atual_kwh']} kWh")
    print(f"  Consumo Decolagem    : {e['consumo_decolagem_kwh']} kWh")
    print(f"  Perdas Estimadas     : {e['perdas_kwh']} kWh")
    print(f"  Consumo Real Total   : {e['consumo_real_kwh']} kWh")
    print(f"  Autonomia Pós-Dec.   : {e['autonomia_pos_decolagem_kwh']} kWh  "
          f"{'Suficiente' if e['suficiente'] else 'Insuficiente'}")

    print("\n ANÁLISE DE IA")
    classificacoes = ia.get("classificacoes", []) if isinstance(ia, dict) else []
    anomalias = ia.get("anomalias", []) if isinstance(ia, dict) else []
    sugestoes = ia.get("sugestoes", []) if isinstance(ia, dict) else []

    print("  Classificações:")
    if classificacoes:
        for c in classificacoes:
            print(f"    • {c}")
    else:
        print("    • Nenhuma classificação disponível")

    print("  Anomalias:")
    if anomalias:
        for a in anomalias:
            print(f"    ⚠ {a}")
    else:
        print("    ⚠ Nenhuma anomalia detectada")

    print("  Sugestões de Risco:")
    if sugestoes:
        for s in sugestoes:
            print(f"    → {s}")
    else:
        print("    → Nenhuma sugestão disponível")



    print(f"\n{'─'*60}")
    print(f"  DECISÃO FINAL: {resultado['decisao']}")
    print(f"{'─'*60}\n")


def gerar_planilha(dados, resultado, energia, ia, arquivo_saida="relatorio_do_foguete.xlsx"):
    linhas_dados = [
        {"Parâmetro": "Timestamp", "Valor": dados["timestamp"]},
        {"Parâmetro": "Temperatura Interna (°C)", "Valor": dados["temp_interna"]},
        {"Parâmetro": "Temperatura Externa (°C)", "Valor": dados["temp_externa"]},
        {
            "Parâmetro": "Integridade Estrutural",
            "Valor": "OK (1)" if dados["integridade_estrutural"] else "FALHA (0)",
        },
        {"Parâmetro": "Nível de Energia (%)", "Valor": dados["nivel_energia"]},
        {"Parâmetro": "Pressão Tanque Ox. (kPa)", "Valor": dados["pressao_tanque_ox"]},
        {"Parâmetro": "Pressão Tanque Comb. (kPa)", "Valor": dados["pressao_tanque_comb"]},
    ]

    for modulo, status in dados["modulos"].items():
        linhas_dados.append(
            {
                "Parâmetro": f"Módulo {modulo.replace('_', ' ').title()}",
                "Valor": "OK (1)" if status else "FALHA (0)",
            }
        )

    df_dados = pd.DataFrame(linhas_dados)

    df_verificacoes = pd.DataFrame(
        [
            {
                "Verificação": check.replace("_", " ").title(),
                "Resultado": "OK" if ok else "FALHA",
            }
            for check, ok in resultado["verificacoes"].items()
        ]
    )

    alertas = resultado["alertas"] if resultado["alertas"] else ["Nenhum alerta"]
    df_alertas = pd.DataFrame({"Alertas": alertas})

    df_energia = pd.DataFrame(
        [
            {"Métrica": "Capacidade Total (kWh)", "Valor": energia["capacidade_total_kwh"]},
            {"Métrica": "Carga Atual (kWh)", "Valor": energia["carga_atual_kwh"]},
            {"Métrica": "Consumo Decolagem (kWh)", "Valor": energia["consumo_decolagem_kwh"]},
            {"Métrica": "Perdas Estimadas (kWh)", "Valor": energia["perdas_kwh"]},
            {"Métrica": "Consumo Real Total (kWh)", "Valor": energia["consumo_real_kwh"]},
            {
                "Métrica": "Autonomia Pós-Decolagem (kWh)",
                "Valor": energia["autonomia_pos_decolagem_kwh"],
            },
            {
                "Métrica": "Status de Energia",
                "Valor": "Suficiente" if energia["suficiente"] else "Insuficiente",
            },
        ]
    )

    df_classificacoes = pd.DataFrame({"Classificações": ia["classificacoes"]})
    df_anomalias = pd.DataFrame({"Anomalias": ia["anomalias"]})
    df_sugestoes = pd.DataFrame({"Sugestões": ia["sugestoes"]})

    df_resumo = pd.DataFrame(
        [
            {"Campo": "Timestamp", "Valor": dados["timestamp"]},
            {"Campo": "Decisão Final", "Valor": resultado["decisao"]},
            {"Campo": "Aprovado", "Valor": "Sim" if resultado["aprovado"] else "Não"},
        ]
    )

    with pd.ExcelWriter(arquivo_saida) as writer:
        df_resumo.to_excel(writer, sheet_name="Resumo", index=False)
        df_dados.to_excel(writer, sheet_name="Dados Coletados", index=False)
        df_verificacoes.to_excel(writer, sheet_name="Verificações", index=False)
        df_alertas.to_excel(writer, sheet_name="Alertas", index=False)
        df_energia.to_excel(writer, sheet_name="Energia", index=False)
        df_classificacoes.to_excel(writer, sheet_name="IA Classificações", index=False)
        df_anomalias.to_excel(writer, sheet_name="IA Anomalias", index=False)
        df_sugestoes.to_excel(writer, sheet_name="IA Sugestões", index=False)

    print(f"Planilha gerada com sucesso: {arquivo_saida}")


dados = coletar_dados()
resultado = verificar_dados(dados)
energia = analise_energetica(dados["nivel_energia"])
ia = analise_ia(dados, resultado, energia)

gerar_relatorio(dados, resultado, energia, ia)
gerar_planilha(dados, resultado, energia, ia)


 RELATORIO DE VERIFICAÇÃO
  2026-03-30T10:20:01.655311

 DADOS COLETADOS
  Temperatura Interna  : 30.0 °C
  Temperatura Externa  : 50.0 °C
  Integridade Estrut.  : OK (1)
  Nível de Energia     : 100.0 %
  Pressão Tanque Ox.   : 270.0 kPa
  Pressão Tanque Comb. : 200.0 kPa
  Módulos Críticos:
   Propulsao : OK (1)
   Navegacao : OK (1)
   Comunicacao : OK (1)
   Controle Termico : OK (1)
   Separacao Estagios : OK (1)

VERIFICAÇÕES DE SEGURANÇA
  Temp Interna : OK
  Temp Externa : OK
  Integridade Estrutural : OK
  Nivel Energia : OK
  Pressao Tanque Ox : OK
  Pressao Tanque Comb : OK
  Modulos Criticos : OK

 ANÁLISE ENERGÉTICA
  Capacidade Total     : 500.0 kWh
  Carga Atual          : 500.0 kWh
  Consumo Decolagem    : 120.0 kWh
  Perdas Estimadas     : 9.6 kWh
  Consumo Real Total   : 129.6 kWh
  Autonomia Pós-Dec.   : 370.4 kWh  Suficiente

 ANÁLISE DE IA
  Classificações:
    • Temperatura interna: NOMINAL
    • Energia: EXCELENTE
    • Todos os módulos críticos: OPERACIONAIS
  